# JFDS Experiment — 03: Ablation Study & Computational Complexity

**Part of a 5-notebook series.** This notebook isolates *why* JFDS works (ablation of its own
internal components — the fairness term, the diversity term, and the gap-squared exponent) and
*what it costs* (asymptotic and empirical runtime analysis of every re-ranking strategy used in
this project).

This is distinct from the baseline comparison in `02_main_experiments` (JFDS vs. MMR vs.
Fairness-only — three *different* re-ranking methods) — here we hold JFDS's own formulation fixed
and remove/modify one internal piece at a time.

Contents:
1. Ablation variant definitions (leave-one-term-out, linear-gap vs. squared-gap)
2. Ablation recommendation generation + per-user metrics
3. Ablation significance testing
4. Big-O complexity derivation for every re-ranking strategy
5. Empirical runtime benchmarks (ms/user, scaling with list length k)

## Setup — load artifacts from `02_main_experiments.ipynb`

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
from IPython.display import display, Markdown

with open('jfds_artifacts.pkl', 'rb') as f:
    ARTIFACTS = pickle.load(f)
globals().update(ARTIFACTS)

plt.rcParams['figure.dpi'] = 110
print(f"Loaded {len(ARTIFACTS)} artifacts from jfds_artifacts.pkl")
print(f"N_USERS={N_USERS}  N_ITEMS={N_ITEMS}  K={K}  POOL_SIZE={POOL_SIZE}")
print("best_lambdas:", best_lambdas)


In [ ]:
# -- Re-declare the core re-ranking + metric functions --
# These are pure functions of (tier, TARGET_SHARE, GENRE_SIM, genre_vec, pop_norm),
# all of which were just loaded from jfds_artifacts.pkl -- no retraining needed.
# Kept byte-identical to their definitions in 02_main_experiments.ipynb so results
# are guaranteed reproducible across notebooks.

def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map  = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])

def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term_fast(cand_idx, selected_idx):
    """Uses pre-computed GENRE_SIM -- no per-call cosine computation."""
    if not selected_idx:
        return 1.0
    return float(1.0 - GENRE_SIM[cand_idx, selected_idx].mean())

def jfds_score(cand_idx, rel_value, selected, tier_counts, step, lam_f, lam_d):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f, lam_d, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)

def mmr_max_sim(cand_idx, selected_idx):
    if not selected_idx:
        return 0.0
    return float(GENRE_SIM[cand_idx, selected_idx].max())

def mmr_score(cand_idx, rel_value, selected, tier_counts, step, lam=MMR_LAMBDA):
    return lam * rel_value - (1 - lam) * mmr_max_sim(cand_idx, selected)

def mmr_rerank(cand_ids, cand_rel, lam=MMR_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, mmr_score, k=k, lam=lam)

def fairness_only_score(cand_idx, rel_value, selected, tier_counts, step, lam_f=FAIR_ONLY_LAMBDA):
    return (1 - lam_f) * rel_value + lam_f * fair_boost(cand_idx, tier_counts, step)

def fairness_only_rerank(cand_ids, cand_rel, lam_f=FAIR_ONLY_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, fairness_only_score, k=k, lam_f=lam_f)

def ild(rec_list):
    """Intra-list diversity: 1 - mean pairwise genre-cosine similarity."""
    if len(rec_list) < 2:
        return 0.0
    idx = np.array(rec_list)
    sub = GENRE_SIM[np.ix_(idx, idx)]
    n = len(idx)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(1.0 - sub[mask].mean())

def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec   = rec_list[:k]
    hits  = len(set(rec) & relevant_set)
    prec  = hits / k
    rec_r = hits / max(1, len(relevant_set))
    dcg   = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    idcg  = sum(g / np.log2(r + 2) for r, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return prec, rec_r, (dcg / idcg if idcg > 0 else 0.0)

def novelty_fairness(rec_list):
    """Mean (1 - normalised popularity): higher = more novel/niche items recommended."""
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))

def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

def aggregate_diversity(rec_lists_dict):
    all_items = set()
    for rec in rec_lists_dict.values():
        all_items.update(rec)
    return len(all_items)

def exposure_vector(rec_lists_dict, n_items=N_ITEMS):
    exp = np.zeros(n_items)
    for rec in rec_lists_dict.values():
        for i in rec:
            exp[i] += 1
    return exp

print('Core re-ranking + metric functions re-declared (greedy_rerank, topk_rerank, jfds_*, '
      'mmr_*, fairness_only_*, ild, precision_recall_ndcg, novelty_fairness, shannon_entropy, '
      'gini, calibration_kl, aggregate_diversity, exposure_vector)')


In [ ]:
# -- Adaptive-lambda schedule (same definition as in 02's "v3 Additions" section) --
DEFAULT_SCHEDULE = dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0)

def schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p=1.0):
    """t_k = step / (k_total - 1); lambda grows from *_start to *_end as t_k^p."""
    t = step / max(1, k_total - 1)
    lam_f = lf_start + (lf_end - lf_start) * (t ** p)
    lam_d = ld_start + (ld_end - ld_start) * (t ** p)
    return lam_f, lam_d

def adaptive_jfds_score(cand_idx, rel_value, selected, tier_counts, step, k_total=K,
                         lf_start=DEFAULT_SCHEDULE['lf_start'], lf_end=DEFAULT_SCHEDULE['lf_end'],
                         ld_start=DEFAULT_SCHEDULE['ld_start'], ld_end=DEFAULT_SCHEDULE['ld_end'],
                         p=DEFAULT_SCHEDULE['p']):
    lam_f, lam_d = schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p)
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def adaptive_jfds_rerank(cand_ids, cand_rel, k=K, **schedule_params):
    params = {**DEFAULT_SCHEDULE, **schedule_params}
    return greedy_rerank(cand_ids, cand_rel, adaptive_jfds_score, k=k, k_total=k, **params)

print('Adaptive-lambda JFDS equation re-declared - default schedule:', DEFAULT_SCHEDULE)


---
## 1. Ablation Study

JFDS's score function is
$$\text{score}(i) = \lambda_u \cdot rel(i) + \lambda_f \cdot \text{fair\_boost}(i) + \lambda_d \cdot \text{diversity}(i, S)$$
with `fair_boost` using a **squared** relative gap, `(gap/target)²` (see `01_theory_and_hypothesis`
for the derivation). Four ablation arms isolate each design choice, holding λ_f, λ_d at the same
Optuna-selected values used for the full JFDS system in `02`:

| Arm | λ_f | λ_d | fair_boost exponent | Tests |
|---|---|---|---|---|
| **JFDS (full)** | optimized | optimized | squared | reference |
| **JFDS − diversity term** | optimized | 0 | squared | does the diversity term matter, holding fairness fixed? |
| **JFDS − fairness term** | 0 | optimized | squared | does the fairness term matter, holding diversity fixed? |
| **JFDS (linear gap)** | optimized | optimized | linear (`gap/target`) | does the squared-gap design choice matter? |

Note: "JFDS − diversity term" is *not* the same as the `FairOnly` baseline in `02` — that baseline
uses a fixed λ_f=0.45 chosen to match JFDS's optimizable upper range, whereas this ablation arm
uses JFDS's own Optuna-selected λ_f. Likewise "JFDS − fairness term" is not the same as `MMR` —
MMR diversifies against the *maximum* similarity to selected items (worst case), while JFDS's own
diversity term (kept here) diversifies against the *mean* similarity (matches how ILD itself is
evaluated). Isolating the static-λ-vs-adaptive-λ choice is a separate, larger comparison already
covered by `AdaptiveJFDS` in `04_significance_and_robustness` — it is not repeated here to avoid
duplicating that computation.

In [ ]:
# -- Ablation variant score functions --
def fair_boost_linear(cand_idx, tier_counts, n_selected):
    """Ablation: linear relative gap instead of the squared gap used in full JFDS."""
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return gap / TARGET_SHARE[t]

def jfds_score_linear_gap(cand_idx, rel_value, selected, tier_counts, step, lam_f, lam_d):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost_linear(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank_linear_gap(cand_ids, cand_rel, lam_f, lam_d, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score_linear_gap, k=k, lam_f=lam_f, lam_d=lam_d)

print('Ablation variant score functions defined (fair_boost_linear, jfds_score_linear_gap, '
      'jfds_rerank_linear_gap)')


In [ ]:
def make_ablation_lists(pools, lf, ld):
    """Generate recommendation lists for every ablation arm, for one base recommender."""
    full_lists, nodiv_lists, nofair_lists, linear_lists = {}, {}, {}, {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        full_lists[u]   = jfds_rerank(cand_ids, cand_rel, lam_f=lf, lam_d=ld)
        nodiv_lists[u]  = jfds_rerank(cand_ids, cand_rel, lam_f=lf, lam_d=0.0)
        nofair_lists[u] = jfds_rerank(cand_ids, cand_rel, lam_f=0.0, lam_d=ld)
        linear_lists[u] = jfds_rerank_linear_gap(cand_ids, cand_rel, lam_f=lf, lam_d=ld)
    return {
        'JFDS (full)': full_lists,
        'JFDS - diversity term': nodiv_lists,
        'JFDS - fairness term': nofair_lists,
        'JFDS (linear gap)': linear_lists,
    }

print('Generating ablation recommendation lists for all users (SVD and BPR) ...')
ablation_lists = {}
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    lf, ld = best_lambdas[model_name]['lam_f'], best_lambdas[model_name]['lam_d']
    ablation_lists[model_name] = make_ablation_lists(pools, lf, ld)
    n_each = {arm: len(lst) for arm, lst in ablation_lists[model_name].items()}
    print(f"  {model_name}: lam_f={lf:.2f} lam_d={ld:.2f}  -- lists per arm: {n_each}")


In [ ]:
# Per-user metrics for every (base_model, ablation arm) combination
abl_rows = []
for model_name, arms in ablation_lists.items():
    for arm_name, rec_lists in arms.items():
        for u, rec in rec_lists.items():
            relevant_set = test_relevant.get(u, set())
            grades       = test_grades.get(u, {})
            p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
            abl_rows.append({
                'base_model': model_name, 'arm': arm_name, 'user': u,
                'precision': p, 'recall': r, 'ndcg': n_ndcg,
                'D': ild(rec), 'F': novelty_fairness(rec),
            })

abl_metrics = pd.DataFrame(abl_rows)
abl_summary = (abl_metrics.groupby(['base_model', 'arm'])[['precision', 'recall', 'ndcg', 'D', 'F']]
               .mean().round(4))
print(f"abl_metrics: {len(abl_metrics):,} rows")
display(abl_summary)


In [ ]:
# -- Bar chart: F / D / NDCG across ablation arms, per base model --
arm_order  = ['JFDS (full)', 'JFDS - diversity term', 'JFDS - fairness term', 'JFDS (linear gap)']
arm_colors = {'JFDS (full)': '#27AE60', 'JFDS - diversity term': '#E67E22',
              'JFDS - fairness term': '#2980B9', 'JFDS (linear gap)': '#8E44AD'}
metrics_bar = ['F', 'D', 'ndcg']
metric_titles = {'F': 'Fairness (novelty)', 'D': 'Diversity (ILD)', 'ndcg': 'NDCG@10 (utility)'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for row, model_name in enumerate(['SVD', 'BPR']):
    for col, metric in enumerate(metrics_bar):
        ax = axes[row][col]
        vals = [abl_metrics[(abl_metrics.base_model == model_name) &
                             (abl_metrics.arm == a)][metric].mean() for a in arm_order]
        bars = ax.bar(range(len(arm_order)), vals, color=[arm_colors[a] for a in arm_order],
                       edgecolor='black')
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(range(len(arm_order)))
        ax.set_xticklabels(arm_order, rotation=25, ha='right', fontsize=8)
        ax.set_title(f'{model_name}: {metric_titles[metric]}', fontsize=10)

plt.suptitle('Ablation Study — JFDS with each internal component removed / modified', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('ablation_bars.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# -- Paired significance: JFDS (full) vs. each ablation arm --
ablation_sig_rows = []
for model_name in ['SVD', 'BPR']:
    full_df = abl_metrics[(abl_metrics.base_model == model_name) &
                           (abl_metrics.arm == 'JFDS (full)')].set_index('user')
    for arm_name in ['JFDS - diversity term', 'JFDS - fairness term', 'JFDS (linear gap)']:
        arm_df = abl_metrics[(abl_metrics.base_model == model_name) &
                              (abl_metrics.arm == arm_name)].set_index('user')
        common = full_df.index.intersection(arm_df.index)
        full_c, arm_c = full_df.loc[common], arm_df.loc[common]
        for metric in ['ndcg', 'D', 'F']:
            a, b = arm_c[metric].values, full_c[metric].values   # a=ablated, b=full
            try:
                w_stat, w_p = stats.wilcoxon(a, b)
            except ValueError:
                w_stat, w_p = np.nan, np.nan
            ablation_sig_rows.append({
                'base_model': model_name, 'ablation_arm': arm_name, 'metric': metric,
                'mean_full_JFDS': b.mean(), 'mean_ablated': a.mean(),
                'mean_delta_full_minus_ablated': (b - a).mean(),
                'wilcoxon_p': w_p, 'n_users': len(common),
            })

ablation_sig_table = pd.DataFrame(ablation_sig_rows).round(5)
display(ablation_sig_table)


**Reading `ablation_sig_table`:** a significant, positive `mean_delta_full_minus_ablated` on a
given metric means removing/modifying that component measurably hurts JFDS on that metric — i.e.
the component earns its place in the equation. A non-significant delta means that component is not
contributing (on this dataset) to that particular metric, which is itself useful evidence for the
theoretical justification in `01_theory_and_hypothesis`.

---
## 2. Computational Complexity Analysis

### Notation

- $P$ = candidate pool size per user (`POOL_SIZE`)
- $k$ = re-ranked list length (`K`)
- $N$ = number of users, $M$ = number of items (`N_ITEMS`)

### Per-user re-ranking cost

`greedy_rerank` runs $k$ steps. At step $s$ (0-indexed), it scans the `remaining` candidate list
(size $\approx P-s$) and calls `score_fn` once per candidate, then removes the chosen item with
`list.remove()` (an $O(P-s)$ scan-and-shift in Python).

The cost of `score_fn` itself depends on the strategy:

| Strategy | `score_fn` cost per candidate | Reason |
|---|---|---|
| Top-K | — (no greedy loop; single `argsort`) | $O(P\log P)$ total |
| Fairness-only | $O(1)$ | `fair_boost` is array/dict lookups only |
| MMR | $O(s)$ | `mmr_max_sim` scans the $s$ already-selected items |
| JFDS (fixed-λ) | $O(s)$ | `diversity_term_fast` averages over the $s$ selected items |
| Adaptive-λ JFDS | $O(s)$ | same diversity term; the λ-schedule lookup itself is $O(1)$ |

Summing the per-step candidate-scan cost $\big(P-s\big)\cdot(\text{score\_fn cost})$ over
$s=0,\dots,k-1$:

$$\text{Fairness-only: } \sum_{s=0}^{k-1} O\!\big((P-s)\cdot 1\big) = O(Pk)$$

$$\text{MMR / JFDS / Adaptive-}\lambda\text{JFDS: } \sum_{s=0}^{k-1} O\!\big((P-s)\cdot s\big)
= O\!\left(Pk^2 - \tfrac{k^3}{3}\right) = O(Pk^2) \quad (k \ll P)$$

The `list.remove()` calls add a further $O(Pk)$ across all strategies that use `greedy_rerank`,
which is dominated by the $O(Pk^2)$ term for MMR/JFDS/Adaptive-JFDS and is the leading term for
Fairness-only (still $O(Pk)$ overall, same order).

**Per-user complexity, ascending order:**
$$\underbrace{O(P\log P)}_{\text{Top-K}} \;\prec\; \underbrace{O(Pk)}_{\text{Fairness-only}}
\;\prec\; \underbrace{O(Pk^2)}_{\text{MMR, JFDS, Adaptive-}\lambda\text{JFDS}}$$

### Full-pipeline complexity (all $N$ users)

$$O\!\big(NP\log P\big) \text{ (Top-K)}, \quad O(NPk) \text{ (Fairness-only)}, \quad
O(NPk^2) \text{ (MMR / JFDS / Adaptive-}\lambda\text{)}$$

plus a **one-time, shared** setup cost of $O(NM)$ to build the SVD/BPR score matrices and
candidate pools, and $O(M^2)$ to precompute `GENRE_SIM` — both amortized once and reused by every
re-ranking strategy, so they do not affect the *relative* ranking above.

### Practical reading

Because $k \ll P$ in this project ($k{=}10$, $P{=}50$), the quadratic-in-$k$ term is not yet
dominant in absolute terms — the empirical benchmarks below confirm this — but it is the term that
will matter most if $k$ is increased for a longer recommendation list (e.g. an infinite-scroll feed),
which is exactly what the empirical scaling-vs-$k$ plot below is designed to detect.

In [ ]:
import time

def time_rerank(rerank_fn, pools, n_sample=300, seed=RANDOM_SEED, **kwargs):
    """Average wall-clock ms/user for one re-ranking strategy over a random user sample."""
    rng = np.random.default_rng(seed)
    sample_u = rng.choice(N_USERS, size=min(n_sample, N_USERS), replace=False)
    t0 = time.perf_counter()
    for u in sample_u:
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        rerank_fn(cand_ids, cand_rel, **kwargs)
    elapsed = time.perf_counter() - t0
    return elapsed / len(sample_u) * 1000  # ms/user

runtime_rows = []
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    lf, ld = best_lambdas[model_name]['lam_f'], best_lambdas[model_name]['lam_d']
    timings = {
        'TopK':          time_rerank(topk_rerank, pools),
        'FairOnly':      time_rerank(fairness_only_rerank, pools, lam_f=FAIR_ONLY_LAMBDA),
        'MMR':           time_rerank(mmr_rerank, pools, lam=MMR_LAMBDA),
        'JFDS':          time_rerank(jfds_rerank, pools, lam_f=lf, lam_d=ld),
        'AdaptiveJFDS':  time_rerank(adaptive_jfds_rerank, pools),
    }
    for method, ms in timings.items():
        runtime_rows.append({'base_model': model_name, 'method': method, 'ms_per_user': ms})
    print(f"{model_name}: " + "  ".join(f"{m}={t:.4f}ms" for m, t in timings.items()))

runtime_df = pd.DataFrame(runtime_rows)
runtime_wide = runtime_df.pivot(index='method', columns='base_model', values='ms_per_user')
display(runtime_wide.round(4))
print(f"\nExtrapolated to all N_USERS={N_USERS:,} users (single-threaded, SVD pools):")
for method in runtime_wide.index:
    total_s = runtime_wide.loc[method, 'SVD'] * N_USERS / 1000
    print(f"  {method:14s}: {total_s:8.1f} s  ({total_s/60:.2f} min)")


In [ ]:
# -- Empirical scaling vs. list length k (confirms O(Pk) vs O(Pk^2)) --
k_values = [5, 10, 15, 20, 25]
scaling_rows = []
model_name = 'SVD'  # representative; BPR pools show the same qualitative scaling
pools = svd_pools
lf, ld = best_lambdas[model_name]['lam_f'], best_lambdas[model_name]['lam_d']
rng = np.random.default_rng(RANDOM_SEED)
sample_u = rng.choice(N_USERS, size=min(150, N_USERS), replace=False)

variants = [
    ('TopK',     topk_rerank,            {}),
    ('FairOnly', fairness_only_rerank,   {'lam_f': FAIR_ONLY_LAMBDA}),
    ('MMR',      mmr_rerank,             {'lam': MMR_LAMBDA}),
    ('JFDS',     jfds_rerank,            {'lam_f': lf, 'lam_d': ld}),
]

for k_test in k_values:
    for method, fn, kwargs in variants:
        t0 = time.perf_counter()
        for u in sample_u:
            cand_ids, cand_rel = pools[u]
            if len(cand_ids) == 0:
                continue
            fn(cand_ids, cand_rel, k=k_test, **kwargs)
        elapsed_ms = (time.perf_counter() - t0) / len(sample_u) * 1000
        scaling_rows.append({'k': k_test, 'method': method, 'ms_per_user': elapsed_ms})

scaling_df = pd.DataFrame(scaling_rows)

fig, ax = plt.subplots(figsize=(8, 5.5))
for method, _, _ in variants:
    sub = scaling_df[scaling_df.method == method]
    ax.plot(sub['k'], sub['ms_per_user'], marker='o', label=method)
ax.set_xlabel('List length k')
ax.set_ylabel('ms / user')
ax.set_title(f'{model_name}: Empirical runtime vs. list length k  (pool size P={POOL_SIZE})')
ax.legend()
plt.tight_layout()
plt.savefig('runtime_vs_k.png', dpi=110, bbox_inches='tight')
plt.show()

display(scaling_df.pivot(index='k', columns='method', values='ms_per_user').round(4))
print('\nIf MMR/JFDS scale as O(k^2) and Fairness-only/Top-K scale sub-quadratically, the '
      'MMR/JFDS curves above should visibly steepen relative to the others as k grows.')


### Complexity + Ablation Summary

- **Ablation** (`ablation_sig_table`): quantifies how much each of JFDS's internal design choices
  — the fairness term, the diversity term, and the squared- vs. linear-gap formulation — actually
  contributes to the metrics it was designed to move, holding the other components fixed.
- **Complexity**: JFDS, MMR, and Adaptive-λ JFDS all share the same $O(NPk^2)$ asymptotic class
  (dominated by the pairwise-similarity diversity term), roughly $k$ times more expensive than the
  $O(NPk)$ Fairness-only baseline, and far more expensive than $O(NP\log P)$ Top-K sorting. The
  empirical benchmarks (`runtime_df`, `scaling_df`) confirm this ordering holds in practice at the
  scale used in this project, and show where it would start to bind (larger $k$) if this were
  deployed with a longer recommendation list.